# Diffusion Models on Country Flag Colors

Most flags have **3 dominant colors**. We encode each flag as a 9-dimensional vector:

    x = [R1, G1, B1,  R2, G2, B2,  R3, G3, B3]

Flag colors come from a small palette (red, white, blue, green, yellow, black),
so flags live on a **low-dimensional manifold** inside 9D space.

We will:
1. Fabricate a dataset of flag-like color triplets
2. Train a tiny diffusion model on it
3. Generate new plausible flag color schemes from pure noise

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ── Canonical flag color palette (normalized RGB in [0, 1]) ──────────────────
RED    = [0.80, 0.10, 0.10]
WHITE  = [0.95, 0.95, 0.95]
BLUE   = [0.05, 0.20, 0.65]
GREEN  = [0.05, 0.55, 0.15]
YELLOW = [0.95, 0.80, 0.05]
BLACK  = [0.08, 0.08, 0.08]

# ── ~30 seed triplets inspired by real flags ─────────────────────────────────
# Each row = [color1, color2, color3] concatenated → 9 numbers
seeds = [
    RED   + WHITE  + BLUE,    # France / Netherlands
    BLUE  + WHITE  + RED,     # Czech Republic style
    GREEN + WHITE  + RED,     # Italy / Hungary
    RED   + YELLOW + RED,     # Spain
    BLACK + RED    + YELLOW,  # Germany
    GREEN + BLACK  + RED,     # Kenya
    RED   + WHITE  + RED,     # Japan-style (repeat)
    BLUE  + YELLOW + BLUE,    # Ukraine
    GREEN + YELLOW + RED,     # Pan-African (Ghana)
    RED   + GREEN  + YELLOW,  # Ethiopia
    WHITE + RED    + WHITE,   # Denmark-style
    BLUE  + WHITE  + BLUE,    # Greece-style
    GREEN + WHITE  + GREEN,   # Nigeria-style
    YELLOW + BLACK + YELLOW,  # Brunei-style
    RED   + WHITE  + GREEN,   # Iran / Mexico
    BLACK + YELLOW + RED,     # Belgium
    BLUE  + RED    + WHITE,   # USA variant
    RED   + BLUE   + WHITE,   # UK variant
    GREEN + RED    + BLACK,   # Palestinian variant
    WHITE + BLUE   + WHITE,   # Argentina-style
    YELLOW + RED   + GREEN,   # Vietnam-style
    RED   + BLACK  + GREEN,   # Libya-style
    BLUE  + WHITE  + GREEN,   # Estonia-style
    GREEN + WHITE  + BLUE,    # Sierra Leone-style
    WHITE + GREEN  + RED,     # Lebanon variant
    BLACK + GREEN  + YELLOW,  # Jamaica
    RED   + WHITE  + BLACK,   # UAE variant
    YELLOW + GREEN + RED,     # Bolivia-style
    BLUE  + GREEN  + WHITE,   # Gabon-style
    RED   + YELLOW + BLACK,   # Vietnam-style dark
]

seeds = np.array(seeds, dtype=np.float32)  # shape (30, 9)

# ── Augment: 12 noisy copies per seed → 360 samples ─────────────────────────
copies = []
for s in seeds:
    for _ in range(12):
        noisy = s + np.random.randn(9).astype(np.float32) * 0.03
        copies.append(np.clip(noisy, 0.0, 1.0))

data = np.stack(copies)          # (360, 9)  —  raw RGB in [0, 1]

# ── Normalize to zero mean, unit std (for stable training) ──────────────────
mu  = data.mean(axis=0)
std = data.std(axis=0).clip(min=1e-6)
data_norm = (data - mu) / std    # shape (360, 9)

print('Dataset shape:', data_norm.shape)

# ── PyTorch DataLoader ───────────────────────────────────────────────────────
tensor_data = torch.from_numpy(data_norm)
loader = DataLoader(TensorDataset(tensor_data), batch_size=64, shuffle=True)

In [ ]:
# ── Fictional country name generator ─────────────────────────────────────────
_PREFIXES = ['Bor', 'Val', 'Mer', 'Cal', 'Ost', 'Nord', 'Sel', 'Tar', 'Vel', 'Dra',
             'Kor', 'Alm', 'Fen', 'Gav', 'Ith', 'Lor', 'Mav', 'Nar', 'Pel', 'Rav']
_SUFFIXES = ['ia', 'ovia', 'estan', 'land', 'ora', 'ania', 'inia', 'ara', 'erra', 'ovia']

def country_name(seed):
    """Deterministic fictional country name from an integer seed."""
    rng = np.random.RandomState(seed)
    return _PREFIXES[rng.randint(len(_PREFIXES))] + _SUFFIXES[rng.randint(len(_SUFFIXES))]

# ── Flag renderer ─────────────────────────────────────────────────────────────
def draw_flag(ax, colors):
    """Draw a horizontal 3-stripe flag on a matplotlib Axes."""
    colors = np.clip(colors, 0, 1)
    for stripe, color in enumerate(colors):
        ax.barh(stripe, 1, color=color, height=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.05, 2.95)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('#888888')
        spine.set_linewidth(0.8)

def show_flags(raw_colors, n=10, title='Flag samples', name_seeds=None):
    """Display n flags as proper 3-stripe banners with country names."""
    fig, axes = plt.subplots(1, n, figsize=(n * 1.3, 2.5))
    fig.suptitle(title, fontsize=11, y=1.02)
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        colors = np.clip(raw_colors[i].reshape(3, 3), 0, 1)
        draw_flag(ax, colors)
        seed = name_seeds[i] if name_seeds is not None else i
        ax.set_title(country_name(seed), fontsize=7, pad=3)
    plt.tight_layout()
    plt.show()

# Show 10 random dataset samples
idx = np.random.choice(len(data), 10, replace=False)
show_flags(data[idx], title='10 flags from the dataset', name_seeds=idx)

## Diffusion: Forward and Reverse

**Forward process** — gradually add noise over T steps:

    x_t = (1 - β) * x_{t-1}  +  β * ε,    ε ~ N(0, I)

After T steps, x_T looks like pure Gaussian noise.

**Reverse process** — a small network predicts the noise residual at each step:

    x_{t-1}  ≈  x_t  -  f_θ(x_t, t/T)

**Training objective** — minimize MSE between predicted and actual residual:

    L = E[ ‖ f_θ(x_t, t/T)  -  (x_t - x_{t-1}) ‖² ]

**Sampling** — start from x_T ~ N(0, I), apply the reverse T times.

In [ ]:
class NoiseMLP(nn.Module):
    """Predicts the noise residual given a noisy flag and the timestep."""
    def __init__(self, data_dim=9, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden),  # flag (9D) + timestep (1D)
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, data_dim),      # predicted residual (9D)
        )

    def forward(self, x, t_norm):
        t_col = t_norm.unsqueeze(1)           # (batch,) → (batch, 1)
        return self.net(torch.cat([x, t_col], dim=1))

net = NoiseMLP().to(device)
print('Parameters:', sum(p.numel() for p in net.parameters()))

In [ ]:
T    = 20     # diffusion timesteps
BETA = 0.10   # noise rate per step
EPOCHS = 300

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

def add_noise_step(x):
    """x_{t} = (1 - beta) * x_{t-1} + beta * noise"""
    return (1 - BETA) * x + BETA * torch.randn_like(x)

losses = []
net.train()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for (x0,) in loader:
        x0 = x0.to(device)

        # Build trajectory x0 → x1 → ... → xT
        xts = [x0]
        for _ in range(T):
            xts.append(add_noise_step(xts[-1]))

        # Train on every step
        step_loss = 0.0
        for t in range(1, T + 1):
            xt     = xts[t]
            x_prev = xts[t - 1]
            t_norm = torch.full((x0.size(0),), t / T, device=device)

            residual_pred = net(xt, t_norm)
            x_prev_pred   = xt - residual_pred

            loss = F.mse_loss(x_prev_pred, x_prev)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            step_loss += loss.item()

        epoch_loss += step_loss / T

    losses.append(epoch_loss / len(loader))
    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | loss: {losses[-1]:.5f}')

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('Training loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def sample_flags(n=10):
    """Generate n flags: start from pure noise, run the reverse process T times."""
    net.eval()
    x = torch.randn(n, 9, device=device)
    for t in reversed(range(1, T + 1)):
        t_norm = torch.full((n,), t / T, device=device)
        x = x - net(x, t_norm)            # subtract predicted residual
    return x.cpu().numpy()

generated_norm = sample_flags(n=10)

# Denormalize back to [0, 1] RGB space
generated = np.clip(generated_norm * std + mu, 0.0, 1.0)

# Use large seeds so names don't collide with the dataset names
show_flags(generated, title='Generated flag color schemes', name_seeds=range(1000, 1010))

In [ ]:
@torch.no_grad()
def denoising_trajectory(n_snaps=7):
    """Show one flag being born from noise, step by step."""
    net.eval()
    x = torch.randn(1, 9, device=device)

    # Collect snapshots at evenly-spaced steps
    snap_every = max(1, T // (n_snaps - 1))
    snaps = [x.cpu().numpy().copy()]
    for t in reversed(range(1, T + 1)):
        t_norm = torch.full((1,), t / T, device=device)
        x = x - net(x, t_norm)
        if t % snap_every == 0:
            snaps.append(x.cpu().numpy().copy())

    # Convert to displayable RGB
    snaps_rgb = [np.clip(s * std + mu, 0, 1).reshape(3, 3) for s in snaps]

    fig, axes = plt.subplots(len(snaps_rgb), 3, figsize=(4, len(snaps_rgb) * 0.8))
    fig.suptitle('Denoising trajectory\n(top = pure noise → bottom = flag)', fontsize=10)
    for i, colors in enumerate(snaps_rgb):
        for j in range(3):
            axes[i, j].set_facecolor(colors[j])
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
        axes[i, 0].set_ylabel(f't={T - i * snap_every}', fontsize=7,
                               rotation=0, labelpad=25)
    plt.tight_layout()
    plt.show()

denoising_trajectory()

In [ ]:
# Optional: PCA projection — do generated flags lie on the same manifold?
from sklearn.decomposition import PCA

gen_norm = sample_flags(n=60)

pca = PCA(n_components=2)
real_2d = pca.fit_transform(data_norm)
gen_2d  = pca.transform(gen_norm)

plt.figure(figsize=(6, 5))
plt.scatter(real_2d[:, 0], real_2d[:, 1], alpha=0.4, s=15, label='Real flags')
plt.scatter(gen_2d[:, 0],  gen_2d[:, 1],  alpha=0.8, s=50, marker='*', label='Generated')
plt.legend()
plt.title('PCA of flag color space')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()